# TestLab Dashboard
Gestione progetti, importazione test, visualizzazione e comparazione.

**Esegui le celle in ordine** — la cella 1 inizializza tutto il sistema.

In [2]:
# ── CELLA 1 — Inizializzazione (esegui sempre per prima) ──────────────────

import sys
from pathlib import Path

# Aggiunge la root del progetto al path così gli import funzionano
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
    
# Il notebook è in testlab-code/testlab/notebooks/
# Il pacchetto 'testlab' è in testlab-code/testlab/
# Quindi ROOT deve puntare a testlab-code/
NB_DIR = Path(".").resolve()
ROOT = NB_DIR.parent.parent  # notebooks/ → testlab/ → testlab-code/

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Esiste testlab/:", (ROOT / "testlab").exists())
    
    
from testlab.core import TestLabDB, FileManager, PluginLoader
from testlab.core import RunMeta, ProjectConfig
import ipywidgets as w
from IPython.display import display, HTML

# Percorsi — adatta se necessario
CODE_ROOT = ROOT                              # repo codice
DATA_ROOT = ROOT.parent / "testlab-data"     # repo dati (DVC)
DB_PATH   = ROOT / "db" / "testlab.sqlite"
SCRIPTS_DIR = ROOT / "scripts"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

db     = TestLabDB(DB_PATH)
loader = PluginLoader(SCRIPTS_DIR)
fm     = FileManager(CODE_ROOT, DATA_ROOT, db)

display(HTML('<div style="background:#EAF3DE;color:#3B6D11;padding:10px 14px;'
             'border-radius:8px;font-size:13px">'
             '✓ TestLab inizializzato correttamente.</div>'))

ROOT: C:\Users\Aquil\Desktop\TestLab\testlab-code
Esiste testlab/: True


## 1 · Gestione progetti e testtype

In [ ]:
# ── CELLA 2 — Crea progetti e testtype ────────────────────────────────────
from testlab.notebooks.widgets import (
    section_title, LogOutput, project_form, testtype_form
)

log_mgmt = LogOutput(height="160px", label="Log gestione")

tabs = w.Tab()
pf   = project_form(db, fm, log_mgmt)
tf   = testtype_form(db, log_mgmt)
tabs.children = [pf, tf]
tabs.set_title(0, "Nuovo progetto")
tabs.set_title(1, "Nuovo testtype")

# Lista progetti esistenti
def refresh_list():
    rows = "".join(
        f'<tr><td style="padding:4px 12px">{p["name"]}</td>'
        f'<td style="padding:4px 12px;color:#888">{p["description"]}</td>'
        f'<td style="padding:4px 12px;font-size:11px;color:#534AB7">{p["dvc_remote"]}</td></tr>'
        for p in db.list_projects()
    )
    return HTML(
        '<table style="font-size:13px;border-collapse:collapse;margin-top:8px">'
        '<tr style="background:#F1EFE8"><th style="padding:4px 12px;text-align:left">'
        'Progetto</th><th style="padding:4px 12px;text-align:left">Descrizione</th>'
        '<th style="padding:4px 12px;text-align:left">DVC remote</th></tr>'
        + rows + '</table>' if rows else
        '<i style="color:#888;font-size:13px">Nessun progetto ancora.</i>'
    )

btn_refresh = w.Button(description="Aggiorna lista", icon="refresh",
                        layout=w.Layout(width="160px"))
out_list    = w.Output()

def on_refresh(_):
    with out_list:
        from IPython.display import clear_output
        clear_output(wait=True)
        display(refresh_list())

btn_refresh.on_click(on_refresh)
on_refresh(None)

display(w.VBox([
    section_title("Gestione progetti"),
    tabs,
    w.HTML('<hr style="margin:16px 0;border-color:#D3D1C7">'),
    section_title("Progetti esistenti", color="gray"),
    w.HBox([btn_refresh]),
    out_list,
    w.HTML('<hr style="margin:16px 0;border-color:#D3D1C7">'),
    log_mgmt.widget,
]))

## 2 · Importazione run

In [6]:
# ── CELLA 3 — Importa un file dati come nuovo run ─────────────────────────
from testlab.notebooks.widgets import section_title, LogOutput, import_run_form

log_import = LogOutput(height="200px", label="Log importazione")

display(w.VBox([
    section_title("Importa run", color="teal"),
    import_run_form(db, fm, log_import),
    w.HTML('<hr style="margin:16px 0;border-color:#D3D1C7">'),
    log_import.widget,
]))

## 3 · Visualizzazione singolo run

In [ ]:
# ── CELLA 4 — Visualizza un singolo run ───────────────────────────────────
from testlab.notebooks.widgets import (
    section_title, LogOutput, info_box,
    make_project_selector, make_testtype_selector, make_run_selector,
)
import plotly.io as pio
pio.renderers.default = "notebook"

log_view = LogOutput(height="120px", label="Log")

proj_dd = make_project_selector(db)
tt_dd   = make_testtype_selector(db, "")
run_dd  = make_run_selector(db, "", "")
out_fig = w.Output()
out_kpi = w.Output()

def update_tt(change):
    p = change["new"]
    tt_dd.options = (["— seleziona —"] +
                     [t["name"] for t in db.list_testtypes(p)]
                     if not p.startswith("—") else [])

def update_runs(change):
    tt = change["new"]
    p  = proj_dd.value
    if tt and not tt.startswith("—") and not p.startswith("—"):
        raw = db.list_runs(p, tt)
        run_dd.options = (
            [("— seleziona —", None)] +
            [(f"{r['run_id']} — {r['condition']}  [{r['date'][:10]}]", r["id"])
             for r in raw]
        )
    else:
        run_dd.options = [("— seleziona —", None)]

proj_dd.observe(update_tt,   names="value")
tt_dd.observe(update_runs,   names="value")

btn_plot = w.Button(description="Visualizza", icon="bar-chart",
                    style=dict(button_color="#534AB7", font_weight="bold"),
                    layout=w.Layout(width="140px"))

def on_plot(_):
    run_id = run_dd.value
    if not run_id:
        log_view.append("Seleziona un run.", "warning")
        return
    try:
        data_file, meta = fm.load_run(run_id)
        proj_cfg = fm.read_project_config(meta.project)
        plugin   = loader.get_for_project(proj_cfg, meta.testtype)
        df       = plugin.read(data_file)
        ok, msg  = plugin.validate(df)
        if not ok:
            log_view.append(f"Validazione: {msg}", "warning")
        fig = plugin.plot_single(df, meta)
        summary = plugin.summary(df, meta)

        with out_fig:
            from IPython.display import clear_output
            clear_output(wait=True)
            fig.show()

        kpi_rows = "".join(
            f'<tr><td style="padding:3px 10px;font-weight:500">{k}</td>'
            f'<td style="padding:3px 10px;font-family:monospace">{v}</td></tr>'
            for k, v in summary.items()
        )
        with out_kpi:
            from IPython.display import clear_output
            clear_output(wait=True)
            display(HTML(
                '<table style="font-size:13px;border-collapse:collapse">'
                '<tr style="background:#F1EFE8"><th style="padding:3px 10px">KPI</th>'
                '<th style="padding:3px 10px">Valore</th></tr>'
                + kpi_rows + '</table>'
            ))
        log_view.append(f"Run {meta.run_id} caricato ({len(df)} righe).", "success")
    except Exception as e:
        log_view.append(str(e), "error")

btn_plot.on_click(on_plot)

display(w.VBox([
    section_title("Visualizzazione singolo run", color="teal"),
    w.HBox([proj_dd, tt_dd]),
    w.HBox([run_dd, btn_plot]),
    w.HBox([
        w.VBox([w.HTML('<b style="font-size:13px">KPI</b>'), out_kpi],
               layout=w.Layout(width="260px")),
        w.VBox([out_fig], layout=w.Layout(flex="1")),
    ]),
    log_view.widget,
]))

## 4 · Comparazione run

In [ ]:
# ── CELLA 5 — Confronta più run dello stesso testtype ─────────────────────
from testlab.notebooks.widgets import (
    section_title, LogOutput,
    make_project_selector, make_testtype_selector, make_run_selector,
)
import pandas as pd

log_cmp  = LogOutput(height="120px", label="Log comparazione")

proj_dd2 = make_project_selector(db)
tt_dd2   = make_testtype_selector(db, "")
runs_ms  = make_run_selector(db, "", "", multi=True)
out_cmp  = w.Output()
out_tbl  = w.Output()

def update_tt2(change):
    p = change["new"]
    tt_dd2.options = (["— seleziona —"] +
                      [t["name"] for t in db.list_testtypes(p)]
                      if not p.startswith("—") else [])

def update_runs2(change):
    tt = change["new"]
    p  = proj_dd2.value
    if tt and not tt.startswith("—") and not p.startswith("—"):
        raw = db.list_runs(p, tt)
        runs_ms.options = [
            (f"{r['run_id']} — {r['condition']}  [{r['date'][:10]}]", r["id"])
            for r in raw
        ]
    else:
        runs_ms.options = []

proj_dd2.observe(update_tt2,   names="value")
tt_dd2.observe(update_runs2,   names="value")

btn_cmp = w.Button(description="Confronta", icon="line-chart",
                   style=dict(button_color="#1D9E75", font_weight="bold"),
                   layout=w.Layout(width="140px"))

def on_compare(_):
    selected_ids = list(runs_ms.value)
    if len(selected_ids) < 2:
        log_cmp.append("Seleziona almeno 2 run (Ctrl+click).", "warning")
        return
    runs_data = []
    summaries = []
    try:
        for rid in selected_ids:
            data_file, meta = fm.load_run(rid)
            proj_cfg = fm.read_project_config(meta.project)
            plugin   = loader.get_for_project(proj_cfg, meta.testtype)
            df       = plugin.read(data_file)
            runs_data.append((df, meta))
            summaries.append(plugin.summary(df, meta))
            log_cmp.append(f"Caricato: {meta.label}", "info")

        # grafico sovrapposto
        proj_cfg = fm.read_project_config(runs_data[0][1].project)
        plugin   = loader.get_for_project(proj_cfg, runs_data[0][1].testtype)
        fig = plugin.plot_compare(runs_data)
        with out_cmp:
            from IPython.display import clear_output
            clear_output(wait=True)
            fig.show()

        # tabella KPI affiancata
        df_kpi = pd.DataFrame(summaries)
        with out_tbl:
            from IPython.display import clear_output
            clear_output(wait=True)
            display(HTML(
                '<b style="font-size:13px">Tabella KPI a confronto</b><br>'
                + df_kpi.to_html(
                    index=False,
                    classes="",
                    border=0,
                ).replace('<table',
                          '<table style="font-size:13px;border-collapse:collapse"')
            ))
        log_cmp.append(f"Confronto completato su {len(selected_ids)} run.", "success")
    except Exception as e:
        log_cmp.append(str(e), "error")

btn_cmp.on_click(on_compare)

display(w.VBox([
    section_title("Comparazione run", color="primary"),
    w.HBox([proj_dd2, tt_dd2]),
    w.HTML('<span style="font-size:12px;color:#888">'
           'Tieni premuto Ctrl (o Cmd su Mac) per selezionare più run</span>'),
    w.HBox([runs_ms, btn_cmp]),
    out_cmp,
    out_tbl,
    log_cmp.widget,
]))

## 5 · Git & DVC

In [9]:
# ── CELLA 6 — Comandi Git e DVC ───────────────────────────────────────────
from testlab.notebooks.widgets import section_title, LogOutput, git_dvc_panel

log_git = LogOutput(height="240px", label="Output comandi")

display(w.VBox([
    section_title("Controllo versione", color="amber"),
    git_dvc_panel(fm, log_git),
    w.HTML('<hr style="margin:16px 0;border-color:#D3D1C7">'),
    log_git.widget,
]))

## 6 · Esplora e filtra run

In [10]:
# ── CELLA 7 — Esplora tutti i run con filtri ──────────────────────────────
from testlab.notebooks.widgets import section_title, make_project_selector
import pandas as pd

proj_flt  = make_project_selector(db, label="Progetto")
cond_flt  = w.Text(description="Condizione:", placeholder="filtra per condizione",
                   style={"description_width": "90px"},
                   layout=w.Layout(width="280px"))
tags_flt  = w.Text(description="Tag:", placeholder="tag1, tag2",
                   style={"description_width": "90px"},
                   layout=w.Layout(width="280px"))
btn_search = w.Button(description="Cerca", icon="search",
                      layout=w.Layout(width="120px"))
out_tbl2   = w.Output()

def on_search(_):
    proj = proj_flt.value
    if proj.startswith("—"):
        with out_tbl2:
            from IPython.display import clear_output
            clear_output(wait=True)
            display(HTML('<i style="color:#888">Seleziona un progetto.</i>'))
        return
    tag_list = [t.strip() for t in tags_flt.value.split(",") if t.strip()]
    cond     = cond_flt.value.strip() or None
    runs     = db.list_runs(proj, condition=cond, tags=tag_list or None)

    if not runs:
        with out_tbl2:
            from IPython.display import clear_output
            clear_output(wait=True)
            display(HTML('<i style="color:#888">Nessun run trovato.</i>'))
        return

    df = pd.DataFrame([
        {
            "id":        r["id"],
            "testtype":  r["testtype"],
            "run_id":    r["run_id"],
            "condizione": r["condition"],
            "data":      r["date"][:10],
            "note":      r["notes"][:40] + "…" if len(r["notes"]) > 40 else r["notes"],
            "tag":       ", ".join(r["tags"]),
        }
        for r in runs
    ])
    with out_tbl2:
        from IPython.display import clear_output
        clear_output(wait=True)
        display(HTML(
            f'<span style="font-size:12px;color:#888">{len(df)} run trovati</span><br>'
            + df.to_html(index=False, border=0).replace(
                '<table',
                '<table style="font-size:13px;border-collapse:collapse"'
            )
        ))

btn_search.on_click(on_search)

display(w.VBox([
    section_title("Esplora run", color="gray"),
    w.HBox([proj_flt, cond_flt, tags_flt, btn_search]),
    out_tbl2,
]))